In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import scanpy as sc
import networkx as nx
import matplotlib.pyplot as plt
from itertools import accumulate
import PyComplexHeatmap as pch
import pandas as pd

In [ ]:
qs = torch.load('results/single_cell/qs.pt',map_location='cpu',weights_only=False)
h  = torch.load('results/single_cell/h.pt',map_location='cpu',weights_only=False)
z  = torch.load('results/single_cell/zs.pt',map_location='cpu',weights_only=False)
Cs  = torch.load('results/single_cell/Cs.pt',map_location='cpu',weights_only=False)

In [ ]:
lung_output = np.argmax(qs[0],axis=1)
kidney_output = np.argmax(qs[1],axis=1)
heart_output = np.argmax(qs[2],axis=1)
liver_output = np.argmax(qs[3],axis=1)

In [ ]:
(qs[3] > 0.1).sum().item()

In [ ]:
print(np.unique(lung_output))
print(np.unique(kidney_output))
print(np.unique(heart_output))
print(np.unique(liver_output))

In [ ]:
lung_label = [str(x) for x in lung_output.detach().numpy()]
kidney_label = [str(x) for x in kidney_output.detach().numpy()]
heart_label = [str(x) for x in heart_output.detach().numpy()]
liver_label = [str(x) for x in liver_output.detach().numpy()]

In [ ]:
lung_output = lung_output.detach().numpy()
kidney_output = kidney_output.detach().numpy()
heart_output = heart_output.detach().numpy()
liver_output = liver_output.detach().numpy()

In [ ]:
def find_missing(remaining_indices,total_length=15):
    full_set = set(range(total_length))
    remaining_set = set(remaining_indices)
    missing = sorted(full_set - remaining_set)
    return missing
ms1 = find_missing(np.unique(lung_output))
ms2 = find_missing(np.unique(kidney_output))
ms3 = find_missing(np.unique(heart_output))
ms4 = find_missing(np.unique(liver_output))

C1 = (Cs[0][1]+Cs[1][0].T)/2
C1= np.delete(C1, ms1, axis=0)
C1= np.delete(C1, ms2, axis=1)
max = C1.max()
min = C1.min()
C1 = (C1 - min)/(max-min)

C2 = (Cs[1][2]+Cs[2][1].T)/2 
C2= np.delete(C2, ms2, axis=0)
C2= np.delete(C2, ms3, axis=1)
max = C2.max()
min = C2.min()
C2 = (C2 - min)/(max-min)

C3 = (Cs[2][3]+Cs[3][2].T)/2 
C3= np.delete(C3, ms3, axis=0)
C3= np.delete(C3, ms4, axis=1)
max = C3.max()
min = C3.min()
C3 = (C3 - min)/(max-min)

C4 = (Cs[3][0]+Cs[0][3].T)/2 
C4= np.delete(C4, ms4, axis=0)
C4= np.delete(C4, ms1, axis=1)
max = C4.max()
min = C4.min()
C4 = (C4 - min)/(max-min)

C5 = (Cs[2][0]+Cs[0][2].T)/2 
C5= np.delete(C5, ms3, axis=0)
C5= np.delete(C5, ms1, axis=1)
max = C5.max()
min = C5.min()
C5 = (C5 - min)/(max-min)

C6 = (Cs[1][3]+Cs[3][1].T)/2 
C6= np.delete(C6, ms2, axis=0)
C6= np.delete(C6, ms4, axis=1)
max = C6.max()
min = C6.min()
C6 = (C6 - min)/(max-min)

In [ ]:
print(C1.shape)
print(C2.shape)
print(C3.shape)
print(C4.shape)
print(C5.shape)
print(C6.shape)

In [ ]:
import anndata as an

In [ ]:
kidney = torch.load('data/single/kidney.pt', map_location='cpu')
lung = torch.load('data/single/lung.pt', map_location='cpu')
heart = torch.load('data/single/heart.pt', map_location='cpu')
liver = torch.load('data/single/liver.pt', map_location='cpu')

In [ ]:
adata = an.AnnData(h[0].detach().numpy())
adata.obs['cluster'] = lung_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
#sc.pl.umap(adata,color='cluster')
sc.pl.umap(adata,color='cluster',title='UMAP of Lung Clusters in Latent Space',save='_lung_emb.png')

In [ ]:
adata = an.AnnData(h[1].detach().numpy())
adata.obs['cluster'] = kidney_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
#sc.pl.umap(adata,color='cluster')
sc.pl.umap(adata,color='cluster',title='UMAP of Kidney Clusters in Latent Space',save='_kidney_emb.png')

In [ ]:
adata = an.AnnData(h[2].detach().numpy())
adata.obs['cluster'] = heart_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
#sc.pl.umap(adata,color='cluster')
sc.pl.umap(adata,color='cluster',title='UMAP of Heart Clusters in Latent Space',save='_heart_emb.png')

In [ ]:
adata = an.AnnData(h[3].detach().numpy())
adata.obs['cluster'] = liver_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
#sc.pl.umap(adata,color='cluster')
sc.pl.umap(adata,color='cluster',title='UMAP of Liver Clusters in Latent Space',save='_liver_emb.png')

In [ ]:
adata = an.AnnData(lung)
adata.obs['cluster'] = lung_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color='cluster')
#sc.pl.umap(adata,color = 'cluster',save='_lung_cluster.png')

In [ ]:
sorted_idx = np.argsort(lung_label)
sorted_adj = lung[sorted_idx, :][:, sorted_idx]
plt.figure(figsize=(8, 6))
sns.heatmap(sorted_adj, cmap='viridis',vmax = 0.6)
plt.savefig('figures/lung_cluster_heatmap.png')
plt.show()
'''

sorted_idx = np.argsort(lung_label)
sorted_adj = lung[sorted_idx, :][:, sorted_idx]

plt.figure(figsize=(10,8))
sns.heatmap(sorted_adj, cmap="viridis", vmin=np.min(lung), vmax=np.max(lung))

sizes = [np.sum(np.array(lung_label) == i) for i in np.unique(np.array(np.array(lung_label)))]
boundaries = list(accumulate(sizes))
for b in boundaries:
    plt.axhline(b, color='red', lw=1)
    plt.axvline(b, color='red', lw=1)
'''

In [ ]:
adata = an.AnnData(kidney)
adata.obs['cluster'] = kidney_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color = 'cluster')
#sc.pl.umap(adata,color = 'cluster',save='_kidney_cluster.png')

In [ ]:
sorted_idx = np.argsort(kidney_label)
sorted_adj = kidney[sorted_idx, :][:, sorted_idx]
plt.figure(figsize=(8, 6))
sns.heatmap(sorted_adj, cmap='viridis',vmax = 0.4)
plt.savefig('figures/kidney_cluster_heatmap.png')
plt.show()
'''
sorted_idx = np.argsort(kidney_label)
sorted_adj = kidney[sorted_idx, :][:, sorted_idx]

plt.figure(figsize=(10,8))
sns.heatmap(sorted_adj, cmap="viridis", vmin=np.min(lung), vmax=np.max(lung))

sizes = [np.sum(np.array(kidney_label) == i) for i in np.unique(np.array(np.array(kidney_label)))]
boundaries = list(accumulate(sizes))
for b in boundaries:
    plt.axhline(b, color='red', lw=1)
    plt.axvline(b, color='red', lw=1)
'''

In [ ]:
adata = an.AnnData(heart)
adata.obs['cluster'] = heart_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
#sc.pp.neighbors(adata,metric='cosine')
sc.tl.umap(adata)
sc.pl.umap(adata,color = 'cluster')
#sc.pl.umap(adata,color = 'cluster',save='_heart_cluster.png')

In [ ]:
sorted_idx = np.argsort(heart_label)
sorted_adj = heart[sorted_idx, :][:, sorted_idx]
plt.figure(figsize=(8, 6))
sns.heatmap(sorted_adj, cmap='viridis',vmax = 0.4)
plt.savefig('figures/heart_cluster_heatmap.png')
plt.show()
'''

sorted_idx = np.argsort(heart_label)
sorted_adj = heart[sorted_idx, :][:, sorted_idx]

plt.figure(figsize=(10,8))
sns.heatmap(sorted_adj, cmap="viridis", vmin=np.min(lung), vmax=np.max(lung))

sizes = [np.sum(np.array(heart_label) == i) for i in np.unique(np.array(np.array(heart_label)))]
boundaries = list(accumulate(sizes))
for b in boundaries:
    plt.axhline(b, color='red', lw=1)
    plt.axvline(b, color='red', lw=1)
'''

In [ ]:
adata = an.AnnData(liver)
adata.obs['cluster'] = liver_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
#sc.pp.neighbors(adata,metric='cosine')
sc.tl.umap(adata)
sc.pl.umap(adata,color = 'cluster')
#sc.pl.umap(adata,color = 'cluster',save='_liver_cluster.png')

In [ ]:
sorted_idx = np.argsort(liver_label)
sorted_adj = liver[sorted_idx, :][:, sorted_idx]
plt.figure(figsize=(8, 6))
sns.heatmap(sorted_adj, cmap='viridis',vmax = 0.4)
plt.savefig('figures/liver_cluster_heatmap.png')
plt.show()
'''

sorted_idx = np.argsort(liver_label)
sorted_adj = liver[sorted_idx, :][:, sorted_idx]

plt.figure(figsize=(10,8))
sns.heatmap(sorted_adj, cmap="viridis", vmin=np.min(liver), vmax=np.max(liver))

sizes = [np.sum(np.array(liver_label) == i) for i in np.unique(np.array(np.array(liver_label)))]
boundaries = list(accumulate(sizes))
for b in boundaries:
    plt.axhline(b, color='red', lw=1)
    plt.axvline(b, color='red', lw=1)
'''

### Fisher’s exact test

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
def fishers_test(organ1,organ2):
    fisher_lk = []
    ps = []
    for i in np.unique(organ1):
        for j in np.unique(organ2):
            a = np.sum((organ1 == i) & (organ2 ==j))
            b = np.sum((organ1 == i) & (organ2 !=j))
            c = np.sum((organ1 != i) & (organ2 ==j))
            d = np.sum((organ1 != i) & (organ2 !=j))
            table = [[a, b],
                     [c, d]]
            _, p = fisher_exact(table,alternative = 'greater')
            ps.append(p)
    ps = np.array(ps)
    _, ps, _, _ = multipletests(ps, alpha=0.05, method='fdr_bh')
    ps = ps.reshape(len(np.unique(organ1)),len(np.unique(organ2)))
    return ps
def star(p):
    if p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    else:
        return ''

In [ ]:
import pandas as pd

In [ ]:
ps = fishers_test(lung_output,kidney_output)
data = pd.DataFrame(C1)
ps = pd.DataFrame(ps)
ps = ps.map(star)

plt.figure(figsize=(5, 4))
sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red', 'weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(kidney_output), 
    yticklabels=np.unique(lung_output), 
)
plt.title("Association between Lung and Kidney Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Kidney')
plt.ylabel('Lung')
plt.tight_layout()
plt.savefig('figures/lung_kidney.png')
plt.show()

In [ ]:
ps = fishers_test(kidney_output,heart_output)
data = pd.DataFrame(C2)
ps = pd.DataFrame(ps)
ps = ps.map(star)

plt.figure(figsize=(5, 4))

sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red', 'weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(heart_output), 
    yticklabels=np.unique(kidney_output), 
)

plt.title("Association between Kidney and Heart Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Heart')
plt.ylabel('Kidney')
plt.tight_layout()
plt.savefig('figures/kidney_heart.png')
plt.show()

In [ ]:
ps = fishers_test(heart_output,liver_output)
data = pd.DataFrame(C3)
ps = pd.DataFrame(ps)
ps = ps.map(star)
plt.figure(figsize=(5, 4))

sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red', 'weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(liver_output), 
    yticklabels=np.unique(heart_output), 
)

plt.title("Association between Heart and Liver Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Liver')
plt.ylabel('Heart')
plt.tight_layout()
plt.savefig('figures/heart_liver.png')
plt.show()

In [ ]:
ps = fishers_test(liver_output,lung_output)
data = pd.DataFrame(C4)
ps = pd.DataFrame(ps)
ps = ps.map(star)
plt.figure(figsize=(5, 4))

sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red','weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(lung_output), 
    yticklabels=np.unique(liver_output), 
)

plt.title("Association between Liver and Lung Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Lung')
plt.ylabel('Liver')
plt.tight_layout()
plt.savefig('figures/liver_lung.png')
plt.show()

In [ ]:
ps = fishers_test(heart_output,lung_output)
data = pd.DataFrame(C5)
ps = pd.DataFrame(ps)
ps = ps.map(star)
print(data.shape)
print(ps.shape)
plt.figure(figsize=(5, 4))

sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red','weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(lung_output), 
    yticklabels=np.unique(heart_output), 
)

plt.title("Association between Heart and Lung Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Lung')
plt.ylabel('Heart')
plt.tight_layout()
plt.savefig('figures/heart_lung.png')
plt.show()

In [ ]:
ps = fishers_test(kidney_output,liver_output)
data = pd.DataFrame(C6)
ps = pd.DataFrame(ps)
ps = ps.map(star)
plt.figure(figsize=(5, 4))

sns.heatmap(
    data,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=ps,
    annot_kws={'fontsize': 10, 'color': 'red','weight': 'bold'},
    fmt='s',
    xticklabels=np.unique(liver_output), 
    yticklabels=np.unique(kidney_output), 
)

plt.title("Association between Kidney and Liver Clusters", fontsize=10)
plt.text(18.5, 1, '* : p <= 0.05', fontsize=10)
plt.text(18.5, 2, '**: p <= 0.01', fontsize=10)
plt.xlabel('Liver')
plt.ylabel('Kidney')
plt.tight_layout()
plt.savefig('figures/kidney_liver.png')
plt.show()

## two organs

In [ ]:
lung_gene = np.argwhere(lung_output == 1).flatten()
kidney_gene = np.argwhere(kidney_output ==8).flatten()
lk = np.intersect1d(lung_gene,kidney_gene)
lk

In [ ]:
lung_gene = np.argwhere(lung_output == 4).flatten()
kidney_gene = np.argwhere(kidney_output ==8).flatten()
lk = np.intersect1d(lung_gene,kidney_gene)
lk

In [ ]:
kidney_gene = np.argwhere(kidney_output == 10).flatten()
heart_gene = np.argwhere(heart_output == 13).flatten()
kh = np.intersect1d(kidney_gene, heart_gene)
kh

In [ ]:
heart_gene = np.argwhere(heart_output == 13).flatten()
lung_gene = np.argwhere((lung_output == 0)|(lung_output == 8)).flatten()
hl = np.intersect1d(heart_gene, lung_gene)
hl

In [ ]:
kidney_gene = np.argwhere(kidney_output == 11).flatten()
heart_gene = np.argwhere(heart_output == 0).flatten()
kh = np.intersect1d(kidney_gene, heart_gene)
kh

In [ ]:
heart_gene = np.argwhere(heart_output == 0).flatten()
liver_gene = np.argwhere(liver_output == 12).flatten()
hl = np.intersect1d(heart_gene, liver_gene)
hl

In [ ]:
heart_gene = np.argwhere(heart_output == 0).flatten()
lung_gene = np.argwhere((lung_output==1)|(lung_output==4)|(lung_output==6)).flatten()
hl = np.intersect1d(heart_gene, lung_gene)
hl

In [ ]:
heart_gene = np.argwhere(heart_output == 11).flatten()
liver_gene = np.argwhere(liver_output == 7).flatten()
hl = np.intersect1d(heart_gene, liver_gene)
hl

## three organs

In [ ]:
kidney_gene = np.argwhere(kidney_output == 10).flatten()
heart_gene = np.argwhere(heart_output == 13).flatten()
lung_gene = np.argwhere((lung_output == 0)|(lung_output == 8)).flatten()
khl = np.intersect1d(np.intersect1d(kidney_gene, heart_gene), lung_gene)
khl

In [ ]:
kidney_gene = np.argwhere(kidney_output == 11).flatten()
heart_gene = np.argwhere(heart_output == 0).flatten()
liver_gene = np.argwhere(liver_output == 12).flatten()
khl = np.intersect1d(np.intersect1d(kidney_gene, heart_gene), liver_gene)
khl

In [ ]:
kidney_gene = np.argwhere(kidney_output == 11).flatten()
heart_gene = np.argwhere(heart_output == 0).flatten()
lung_gene = np.argwhere((lung_output==1)|(lung_output==4)|(lung_output==6))
khl = np.intersect1d(np.intersect1d(kidney_gene, heart_gene), lung_gene)
khl

## four organs

In [ ]:
lung_gene = np.argwhere(lung_output == 14).flatten()
kidney_gene = np.argwhere(kidney_output == 14).flatten()
heart_gene = np.argwhere(heart_output == 8).flatten()
liver_gene = np.argwhere(liver_output == 1).flatten()
lk = np.intersect1d(lung_gene, kidney_gene)
kh = np.intersect1d(kidney_gene, heart_gene)
hl = np.intersect1d(heart_gene, liver_gene)
ll = np.intersect1d(liver_gene, lung_gene)
lkhl = np.intersect1d(np.intersect1d(np.intersect1d(kidney_gene, heart_gene), lung_gene), liver_gene)

In [ ]:
lung_gene = np.argwhere((lung_output==1)|(lung_output==4)|(lung_output==6))
kidney_gene = np.argwhere(kidney_output == 11).flatten()
heart_gene = np.argwhere(heart_output == 0).flatten()
liver_gene = np.argwhere(liver_output == 12).flatten()
lkhl = np.intersect1d(np.intersect1d(np.intersect1d(kidney_gene, heart_gene), lung_gene), liver_gene)
lkhl

## name

In [ ]:
lung_gene

In [ ]:
isolated = torch.load('data/single/isolated.pt')

In [ ]:
adata = sc.read_h5ad('../single_cell_data/adata.h5ad')
adata

In [ ]:
keep_idx = [i for i in range(adata.n_vars) if i not in isolated]
adata = adata[:, keep_idx].copy()
adata

In [ ]:
lung = adata.var_names[list(lung_gene)]     
kidney = adata.var_names[list(kidney_gene)]  
heart = adata.var_names[list(heart_gene)]  
liver = adata.var_names[list(liver_gene)]
lung_kidney = adata.var_names[lk]
kidney_heart = adata.var_names[kh]
heart_liver = adata.var_names[hl]
liver_lung = adata.var_names[ll]
lung_kidney_heart_liver = adata.var_names[lkhl]

df = pd.DataFrame({
    'lung': pd.Series(lung),
    'kidney': pd.Series(kidney),
    'heart': pd.Series(heart),
    'liver': pd.Series(liver),
    'lung_kidney': pd.Series(lung_kidney),
    'kidney_heart': pd.Series(kidney_heart),
    'heart_liver': pd.Series(heart_liver),
    'liver_lung': pd.Series(liver_lung),
    'lung_kidney_heart_liver': pd.Series(lung_kidney_heart_liver)
    
})
df.to_csv("gene.csv", index=False)

In [ ]:
len(lung_gene)

In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('single_cell/data/differential_expression/heart/Reichart_2022_all_deg_tval.csv',usecols=['Unnamed: 0','log2FoldChange_mesenchymal', 'pvalue_mesenchymal'])
df1

In [ ]:
gene_df = pd.DataFrame(genes, columns=['Unnamed: 0'])
print(gene_df)

In [ ]:
df1.merge(gene_df, on='Unnamed: 0', how='inner')

In [ ]:
len(lkhl)

In [ ]:
idx = lung_gene

genes = adata.var_names[idx].to_numpy()
adj = torch.load('data/single/lung.pt')
adj = adj[np.ix_(idx, idx)]
print(adj.shape)
edges = []
for i in range(len(genes)):
    for j in range(i, len(genes)):
        weight = adj[i, j]
        if weight != 0:
            edges.append([genes[i], genes[j], weight])

df_edges = pd.DataFrame(edges, columns=["Source", "Target", "Weight"])
df_edges.to_csv("data/single/lung_gene.csv", index=False)

In [ ]:
adata.var_names[idx]

In [ ]:
g = torch.load('../data/single/lung_topk.pt',weights_only = False)
df = pd.DataFrame(lung_label, columns=['cluster'])
order = df.sort_values('cluster').index.tolist()

data_sorted = pd.DataFrame(g).iloc[order, order].reset_index(drop=True)
data_sorted.columns = range(len(data_sorted.columns)) 
df_sorted = df.iloc[order].reset_index(drop=True)

row_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=True, legend=True, cmap='tab20'),
    axis=0
)
col_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=True, legend=True, cmap='tab20'),
    axis=1
)

plt.figure(figsize=(7, 6))
cm = pch.ClusterMapPlotter(
    data=data_sorted,
    top_annotation=col_ha,
    left_annotation=row_ha,
    row_cluster=False,
    col_cluster=False,
    cmap='Reds',
    vmin=0,
    vmax=1,
    row_names_side='left',
    rasterized=True
)
plt.savefig("heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()